<a href="https://colab.research.google.com/github/Shamsfathalla/FlyRank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shamsfathalla/FlyRank-Starter-Notebooks/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row means: One content page on one specific calendar day (content_hash_id × report_date).

Table used: fact_content_daily_performance from FlyRank/internship-warehouse.

Time window: Mid-panel month month = '2026-03' (2026-03-01 to 2026-03-31), reserving June 2026 as the sealed holdout set.

Target to predict: is_underperforming_opportunity (1 if page has high exposure but low engagement: gsc_impressions > 50 and gsc_clicks == 0, else 0).

Excluded field: gsc_clicks. Why: It directly constructs the target definition; including it creates 100% target leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
import pandas as pd
from google.colab import userdata

# Load token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Connect DuckDB to Hugging Face
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
table_path = f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')"

print("Warehouse connection verified and ready.")

Warehouse connection verified and ready.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features:

gsc_impressions: Top-of-funnel search visibility.

gsc_avg_position: Search ranking depth.

ga4_sessions: Total inbound traffic landing on the page.

ga4_engaged_sessions: User engagement indicator on site.

scroll_events: On-page interaction depth.

Label / Proxy: is_underperforming_opportunity (Binary flag: high impressions with zero search clicks).

Context: content_hash_id, client_hash_id, report_date, month.

Excluded: gsc_clicks (creates target leakage with the opportunity proxy).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

gsc_impressions: knowable at the decision moment because Google logs search visibility daily before we do our review.

gsc_avg_position: knowable at the decision moment because Google records ranking depth every day.

ga4_sessions: knowable at the decision moment because Analytics tracks daily visits as they happen.

ga4_engaged_sessions: knowable at the decision moment because bounce tracking is processed right after the user leaves.

scroll_events: knowable at the decision moment because scrolling is tracked in real-

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --- 1. The Grain (One row = one page per day) ---
q1 = f"""
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT content_hash_id || report_date) AS unique_rows
FROM {table_path}
WHERE month = '2026-03'
"""
print("Grain: Does total rows match unique page-days?")
print(con.sql(q1).df().to_string(index=False), "\n")

Grain: Does total rows match unique page-days?


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  unique_rows
    9841378      9841378 



In [ ]:
# Row Count & Date Span
q2 = f"""
SELECT COUNT(*) AS march_rows, MIN(report_date) AS first_day, MAX(report_date) AS last_day
FROM {table_path}
WHERE month = '2026-03'
"""
print("Span: Exactly how much data is in our March slice?")
print(con.sql(q2).df().to_string(index=False), "\n")


Span: Exactly how much data is in our March slice?
 march_rows  first_day   last_day
    9841378 2026-03-01 2026-03-31 



In [ ]:
# Availability
q3 = f"""
SELECT COUNT(*) AS clean_rows
FROM {table_path}
WHERE month = '2026-03'
  AND (gsc_impressions IS NOT NULL) IS TRUE
  AND (gsc_avg_position IS NOT NULL) IS TRUE
  AND (ga4_sessions IS NOT NULL) IS TRUE
  AND (ga4_engaged_sessions IS NOT NULL) IS TRUE
  AND (scroll_events IS NOT NULL) IS TRUE
"""
print("Availability: How many rows survive with no missing features?")
print(con.sql(q3).df().to_string(index=False))

Availability: How many rows survive with no missing features?


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 clean_rows
    2082695


In [ ]:
# Build Features & Spring the Trap
print("The Leakage Trap")

# Pull a clean sample of 10,000 rows
q_frame = f"""
SELECT gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, ga4_engaged_sessions, scroll_events
FROM {table_path}
WHERE month = '2026-03'
  AND (gsc_impressions IS NOT NULL) IS TRUE
  AND (gsc_avg_position IS NOT NULL) IS TRUE
LIMIT 10000
"""
df = con.sql(q_frame).df()

# The proxy label: High impressions (>50) but exactly 0 clicks
df['is_opportunity'] = ((df['gsc_impressions'] > 50) & (df['gsc_clicks'] == 0)).astype(int)

# Honest features
X = df[['gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'scroll_events']].copy()
y = df['is_opportunity'].copy()

# Spring the trap: Add clicks to the features
X['leaked_clicks'] = df['gsc_clicks']

# Prove it ruins the model (Correlation jumps drastically)
print(f"Danger: Correlation with leaked feature: {X['leaked_clicks'].corr(y):.4f}")

# Delete it and keep the honest number
X = X.drop(columns=['leaked_clicks'])
print("Trap deleted. The feature frame is honest again.")

The Leakage Trap
Danger: Correlation with leaked feature: -0.0902
Trap deleted. The feature frame is honest again.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What this data can never tell me: Why a user ignored the page.
This data shows symptoms, not root causes. If a page gets 1,000 impressions and zero clicks, the data cannot tell me if the page title was badly written, or if the user's search intent just didn't match our topic. We only know the page is being ignored, not why it is being ignored.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.